# 03 Exploratory Data Analysis

Explore the two cleaned datasets to surface distributions, trends, and hazard signals relevant to planetary defense.

**Guiding questions**
1. How are close-approach distances and velocities distributed? Any outliers?
2. Are close approaches increasing or flat over time (2015–2035)?
3. What share of catalogued NEAs are flagged Potentially Hazardous (PHA)? How do their size and orbital profiles differ?
4. Which asteroids appearing in the close-approach log are also PHAs?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

close = pd.read_csv(PROCESSED_DIR / 'close_approaches_clean.csv', parse_dates=['close_approach_date'])
nea   = pd.read_csv(PROCESSED_DIR / 'near_earth_asteroids_clean.csv', parse_dates=['first_obs', 'last_obs'])
print('close:', close.shape, '| nea:', nea.shape)


## 1. Close-approach distance and velocity distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(close['dist_lunar'], bins=60, ax=axes[0], color='steelblue')
axes[0].set_title('Close-approach distance (lunar distances)')
axes[0].set_xlabel('Distance (LD, 1 LD = Earth-Moon distance)')
sns.histplot(close['velocity_km_s'], bins=60, ax=axes[1], color='indianred')
axes[1].set_title('Relative velocity at approach')
axes[1].set_xlabel('km/s')
plt.tight_layout()
plt.show()

print('Distance (LD) summary:')
print(close['dist_lunar'].describe().round(2))
print('\nVelocity (km/s) summary:')
print(close['velocity_km_s'].describe().round(2))


## 2. Close approaches over time

In [ ]:
yearly = close.groupby('approach_year').size().rename('approaches')
fig, ax = plt.subplots(figsize=(11, 4))
yearly.plot(kind='bar', ax=ax, color='slateblue')
ax.set_title('Close approaches per year (2015–2035)')
ax.set_ylabel('Number of approaches')
ax.set_xlabel('Year')
plt.tight_layout()
plt.show()
yearly.describe().round(2)


## 3. Top 15 closest approaches in the window

In [ ]:
closest = close.nsmallest(15, 'dist_lunar')[
    ['designation', 'close_approach_date', 'dist_lunar', 'dist_km', 'velocity_km_s', 'is_future']
].reset_index(drop=True)
closest


## 4. PHA share and size-category breakdown (NEA catalogue)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
nea['pha'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'firebrick'])
axes[0].set_title(f"PHA flag — {nea['pha'].mean()*100:.2f}% of catalogue")
axes[0].set_xticklabels(['Non-hazardous', 'PHA'], rotation=0)

size_order = nea['size_category'].value_counts().index
sns.countplot(data=nea, y='size_category', order=size_order, ax=axes[1], color='slategray')
axes[1].set_title('Size categories')
plt.tight_layout()
plt.show()


## 5. PHA vs non-PHA — diameter and MOID

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.boxplot(data=nea, x='pha', y='diameter_km', ax=axes[0], showfliers=False)
axes[0].set_yscale('log')
axes[0].set_title('Diameter (log) by PHA flag')

sns.boxplot(data=nea, x='pha', y='moid_lunar_distances', ax=axes[1], showfliers=False)
axes[1].set_title('MOID (lunar distances) by PHA flag')
plt.tight_layout()
plt.show()

nea.groupby('pha')[['diameter_km', 'moid_lunar_distances', 'h']].median().round(3)


## 6. Orbital element distributions (eccentricity, inclination, semi-major axis)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(nea['e'], bins=50, ax=axes[0], color='teal'); axes[0].set_title('Eccentricity (e)')
sns.histplot(nea['i'], bins=50, ax=axes[1], color='darkorange'); axes[1].set_title('Inclination (deg)')
sns.histplot(nea['a'], bins=50, ax=axes[2], color='purple'); axes[2].set_title('Semi-major axis (AU)')
plt.tight_layout()
plt.show()


## 7. Cross-dataset — which close approachers are PHAs?

In [ ]:
merged = close.merge(
    nea[['pdes', 'pha', 'diameter_km', 'size_category', 'class']],
    left_on='designation', right_on='pdes', how='left'
)

matched = merged['pdes'].notna().sum()
print(f'{matched:,} / {len(merged):,} close-approach rows matched the NEA catalogue ({matched/len(merged)*100:.1f}%)')
print(f"PHA close approaches: {int(merged['pha'].sum())}")
print(f"Closest PHA approach (LD): {merged.loc[merged['pha'] == True, 'dist_lunar'].min():.3f}")

pha_approaches = (
    merged[merged['pha'] == True]
    .nsmallest(10, 'dist_lunar')[
        ['designation', 'close_approach_date', 'dist_lunar', 'velocity_km_s', 'diameter_km', 'size_category']
    ]
    .reset_index(drop=True)
)
pha_approaches


### Takeaways

- Close-approach distances are right-skewed; most events pass well beyond the Moon, with a long tail of very close flybys.
- Annual approach counts are roughly stable across the 20-year window — no strong trend visible at this resolution.
- Roughly 5% of catalogued NEAs are PHAs; they skew larger (bigger diameter, lower H) and, by definition, closer MOID.
- The close-approach log is dominated by non-PHA objects, but the closest individual passes include several PHAs — the key signal for the dashboard.